# PowerPlus — XGBoost User Input Prediction Model (Fewer User Features)

This notebook is the **user-input prediction stage** for the selected XGBoost model.

### Dashboard-friendly design
- The trained XGBoost model keeps its full feature set internally.
- The user enters only the **most necessary household and equipment features**.
- All other trained features are automatically filled from the selected city's latest profile/history.
- Calendar features are derived automatically from the forecast start date.
- Lag and rolling electricity-demand features are taken from historical demand automatically; the user does **not** enter them.
- Categorical inputs are **case-insensitive** and use numbered dropdown-style choices.
- Produces 1-day, 7-day, and 30-day forecasts.
- Creates CSV outputs ready for **Power BI**, including a prediction summary and appliance sensitivity.

### User-facing inputs
**Location:** City  
**Forecast:** Forecast start date  
**Household/home:** Total residents, Covered Area, Number of rooms, House age, Ceiling Type, Roof Type  
**Key equipment:** Air Conditioners, Air Coolers, Refrigerators, Washing Machines, Ceiling Fans, Water Pumps, Electric Heaters, Electric Cooker, Geysers, LED Bulbs

**Important:** the `.pkl` must be the complete trained XGBoost artifact containing the prediction pipeline. A feature-name array alone is not sufficient.


In [17]:
# 1. IMPORTS
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)


In [18]:
# 2. LOAD THE TRAINED XGBOOST MODEL + FEATURE-ENGINEERED DATA + DAILY WEATHER
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    for p in candidates:
        if (p / 'processed_data').exists():
            return p
    return Path.cwd()

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
MODEL_PKL = PROCESSED_DIR / 'powerplus_xgboost_forecast_model.pkl'
MODEL_DATA = PROCESSED_DIR / 'powerplus_model_data.csv'
WEATHER_FILE = PROCESSED_DIR / 'daily_weather.csv'

for required in [MODEL_PKL, MODEL_DATA, WEATHER_FILE]:
    if not required.exists():
        raise FileNotFoundError(f'Required file not found: {required}')

artifact = joblib.load(MODEL_PKL)
if not isinstance(artifact, dict) or 'pipeline' not in artifact or 'feature_columns' not in artifact:
    raise TypeError('The .pkl must contain at least pipeline and feature_columns.')

pipeline = artifact['pipeline']
feature_columns = list(artifact['feature_columns'])
TARGET = artifact.get('target', 'electricity_kwh')
DATE_COL = artifact.get('date_column', 'date')
CITY_COL = artifact.get('city_column', 'city')
HOUSE_COL = None  # House is completely removed from training and user prediction.
numeric_features = list(artifact.get('numeric_features', []))
categorical_features = list(artifact.get('categorical_features', []))

df = pd.read_csv(MODEL_DATA)
weather_df = pd.read_csv(WEATHER_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')

weather_date_col = next((c for c in ['date','Date','DATE'] if c in weather_df.columns), None)
weather_city_col = next((c for c in ['city','City','CITY'] if c in weather_df.columns), None)
if weather_date_col is None or weather_city_col is None:
    raise ValueError(f'daily_weather.csv needs City and date. Found: {list(weather_df.columns)}')

weather_df[weather_date_col] = pd.to_datetime(weather_df[weather_date_col], errors='coerce')
df['_merge_city'] = df[CITY_COL].astype(str).str.strip().str.casefold()
df['_merge_date'] = df[DATE_COL].dt.normalize()
weather_df['_merge_city'] = weather_df[weather_city_col].astype(str).str.strip().str.casefold()
weather_df['_merge_date'] = weather_df[weather_date_col].dt.normalize()

weather_value_cols = [c for c in weather_df.columns if c not in {weather_city_col, weather_date_col, '_merge_city', '_merge_date'}]
w = weather_df[['_merge_city','_merge_date'] + weather_value_cols].drop_duplicates(
    ['_merge_city','_merge_date'], keep='last'
)
rename_map = {c: f'{c}_weather' for c in weather_value_cols if c in df.columns}
w = w.rename(columns=rename_map)
df = df.merge(w, on=['_merge_city','_merge_date'], how='left', validate='m:1')
df = df.drop(columns=['_merge_city','_merge_date'])
weather_value_cols = [rename_map.get(c,c) for c in weather_value_cols]

df = df.dropna(subset=[DATE_COL, TARGET]).sort_values([CITY_COL, DATE_COL]).reset_index(drop=True)

# House must NOT be a model feature in the updated XGBoost artifact.
if HOUSE_COL in feature_columns:
    raise RuntimeError(
        'This notebook requires the UPDATED XGBoost .pkl trained with House removed. '
        'Run the updated XGBoost training notebook first.'
    )

if not numeric_features and not categorical_features:
    numeric_features = [c for c in feature_columns if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    categorical_features = [c for c in feature_columns if c not in numeric_features]

print('Model:', MODEL_PKL)
print('Training/history data:', MODEL_DATA)
print('Weather:', WEATHER_FILE)
print('Target:', TARGET)
print('Total model features:', len(feature_columns))
print('House in model features: REMOVED')
print('History:', df[DATE_COL].min().date(), 'to', df[DATE_COL].max().date())
print('Merged weather columns:', weather_value_cols)


Model: d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_xgboost_forecast_model.pkl
Training/history data: d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_model_data.csv
Weather: d:\Project-Electricity-Demand-Forecasting\processed_data\daily_weather.csv
Target: electricity_kwh
Total model features: 95
House in model features: REMOVED
History: 2023-08-15 to 2024-11-27
Merged weather columns: ['temperature_mean_weather', 'humidity_mean_weather', 'dew_mean_weather', 'wind_speed_mean_weather', 'wind_direction_mean_weather', 'pressure_mean_weather', 'solar_radiation_mean_weather', 'temperature_max_weather', 'humidity_max_weather', 'wind_speed_max_weather', 'solar_radiation_max_weather', 'uv_index_max_weather', 'precipitation_sum_weather', 'solar_energy_sum_weather']


In [19]:
# 3. BUILD AND SAVE THE COMPLETE INPUT SCHEMA
# IMPORTANT: Run this cell BEFORE the user-input cell.
# It creates the exact 'label' column used later.

schema_rows = []

for c in feature_columns:
    if c in df.columns:
        s = df[c]
    else:
        s = pd.Series(dtype="object")

    if c in numeric_features:
        model_type = "numeric"
        python_input_type = "float"
        value_type = "number"
    elif c in categorical_features:
        model_type = "categorical"
        python_input_type = "str"
        value_type = "string/category"
    else:
        model_type = "numeric" if pd.api.types.is_numeric_dtype(s) else "categorical"
        python_input_type = "float" if model_type == "numeric" else "str"
        value_type = "number" if model_type == "numeric" else "string/category"

    vals = s.dropna()
    unique_vals = vals.astype(str).drop_duplicates().tolist()

    min_value = np.nan
    max_value = np.nan
    if model_type == "numeric" and len(vals):
        num = pd.to_numeric(vals, errors="coerce").dropna()
        if len(num):
            min_value = num.min()
            max_value = num.max()

    schema_rows.append({
        "label": c,
        "model_type": model_type,
        "pandas_dtype": str(s.dtype),
        "python_input_type": python_input_type,
        "value_type": value_type,
        "unique_count": int(vals.nunique(dropna=True)),
        "sample_values": " | ".join(unique_vals[:10]),
        "min": min_value,
        "max": max_value,
        "allowed_values_first_200": (
            " | ".join(sorted(unique_vals, key=lambda x: x.casefold())[:200])
            if model_type == "categorical" else ""
        ),
    })

schema_df = pd.DataFrame(schema_rows)

# Safety check: this prevents the KeyError: 'label'
required_schema_columns = [
    "label", "model_type", "pandas_dtype",
    "python_input_type", "value_type"
]
missing_schema_columns = [
    c for c in required_schema_columns if c not in schema_df.columns
]
if missing_schema_columns:
    raise RuntimeError(
        f"Schema creation failed. Missing columns: {missing_schema_columns}"
    )

SCHEMA_FILE = PROCESSED_DIR / "powerplus_user_input_schema.csv"
schema_df.to_csv(SCHEMA_FILE, index=False)

print("SCHEMA CREATED SUCCESSFULLY")
print("=" * 72)
print(f"Total features : {len(schema_df)}")
print(f"Numeric        : {(schema_df['model_type'] == 'numeric').sum()}")
print(f"Categorical    : {(schema_df['model_type'] == 'categorical').sum()}")
print(f"Saved          : {SCHEMA_FILE}")

display(schema_df)


SCHEMA CREATED SUCCESSFULLY
Total features : 95
Numeric        : 81
Categorical    : 14
Saved          : d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_user_input_schema.csv


,label,model_type,pandas_dtype,python_input_type,value_type,unique_count,sample_values,min,max,allowed_values_first_200
0,city,categorical,object,str,string/category,6,Islamabad | Karachi | Lahore | Multan | Peshawar | Skardu,NaN,NaN,Islamabad | Karachi | Lahore | Multan | Peshawar | Skardu
1,House,categorical,object,str,string/category,59,House#50 | House#41 | House#42 | House#48 | House#49 | House#43 | House#44 | House#46 | House#45 | House#47,NaN,NaN,House#1 | House#10 | House#11 | House#12 | House#13 | House#14 | House#15 | House#16 | House#17 | House#18 | House#1...
2,City,categorical,object,str,string/category,6,Islamabad | Karachi | Lahore | Multan | Peshawar | Skardu,NaN,NaN,Islamabad | Karachi | Lahore | Multan | Peshawar | Skardu
3,Owner/Rented,categorical,object,str,string/category,3,Owner | Rented | Residing as tenant,NaN,NaN,Owner | Rented | Residing as tenant
4,No. of people (Temp+Perm),numeric,float64,float,number,12,8.0 | 6.0 | 4.0 | 3.0 | 2.0 | 7.0 | 9.0 | 5.0 | 11.0 | 0.0,0.0,25.0,
...,...,...,...,...,...,...,...,...,...,...
90,wind_speed_max_weather,numeric,float64,float,number,113,4.0 | 4.7 | 3.6 | 5.8 | 6.5 | 2.9 | 7.9 | 6.1 | 17.6 | 8.3,0.0,55.4,
91,solar_radiation_max_weather,numeric,float64,float,number,1,0.0,0.0,0.0,
92,uv_index_max_weather,numeric,int64,float,number,1,0,0.0,0.0,
93,precipitation_sum_weather,numeric,float64,float,number,27,0.0 | 1.2 | 0.1 | 0.3 | 0.7 | 1.6 | 3.1 | 3.4 | 0.6 | 1.1,0.0,4.5,


In [20]:
# 4. DEFINE THE FEW USER-FACING FEATURES
# The trained XGBoost model keeps its full feature set internally.
# Only these practical fields are entered by the user; everything else is auto-filled.

USER_INPUT_FEATURE_LABELS = [
    # Home / household
    'No. of people (Temp+Perm)',
    'Covered Area',
    'No. of rooms',
    'house_age',
    'Ceiling Type',
    'Roof Type',

    # Key equipment
    'Air Conditioners',
    'Air Coolers',
    'Refrigerators',
    'Washing Machines',
    'Celling Fans',  # accepts the training spelling; matching is case/spelling tolerant
    'Water Pumps',
    'Electric heaters',  # accepts case/spelling variants
    'Electric Cooker',
    'Geysers',
    'LED Bulbs',
]

# Case/spelling-tolerant matching to the actual trained feature names.
def norm_name(x):
    return ''.join(ch.lower() for ch in str(x) if ch.isalnum())

feature_norm = {norm_name(c): c for c in feature_columns}
USER_FEATURES = []
for label in USER_INPUT_FEATURE_LABELS:
    n = norm_name(label)
    if n in feature_norm and feature_norm[n] not in USER_FEATURES:
        USER_FEATURES.append(feature_norm[n])

# Detect ALL appliance/equipment columns in the trained model for internal calculations.
ALL_APPLIANCE_KEYWORDS = [
    'aircondition', 'aircool', 'refriger', 'washingmachine', 'ledbulb',
    'tubel', 'fan', 'waterdispenser', 'waterpump', 'electriccooker',
    'electricheater', 'electriciron', 'sewingmachine', 'microwave',
    'geyser', 'ups', 'electronicdevice'
]
ALL_APPLIANCE_COLUMNS = [
    c for c in feature_columns
    if any(k in norm_name(c) for k in ALL_APPLIANCE_KEYWORDS)
]

# Only the selected equipment is shown to the user.
appliance_columns = [c for c in USER_FEATURES if c in ALL_APPLIANCE_COLUMNS]

print('USER-FACING INPUTS')
print('=' * 72)
print('Location : City')
print('Forecast : Forecast start date')
print('Home    :', [c for c in USER_FEATURES if c not in appliance_columns])
print('Equipment:', appliance_columns)
print(f'Total user-entered feature fields: {len(USER_FEATURES)}')


USER-FACING INPUTS
Location : City
Forecast : Forecast start date
Home    : ['No. of people (Temp+Perm)', 'Covered Area', 'No. of rooms', 'house_age', 'Ceiling Type', 'Roof Type']
Equipment: ['Air Conditioners', 'Air Coolers', 'Refrigerators', 'Washing Machines', 'Celling Fans', 'Water Pumps', 'Electric heaters', 'Electric Cooker', 'Geysers', 'LED Bulbs']
Total user-entered feature fields: 16


In [21]:
# 5. CASE-INSENSITIVE INPUT HELPERS
def canonical_category(label, raw, allowed):
    '''Return the exact training category while accepting any letter case.'''
    text = str(raw).strip()
    lookup = {str(v).strip().casefold(): v for v in allowed}
    if text.casefold() in lookup:
        return lookup[text.casefold()]
    # Friendly whitespace normalization
    compact = ' '.join(text.split()).casefold()
    lookup2 = {' '.join(str(v).split()).casefold(): v for v in allowed}
    if compact in lookup2:
        return lookup2[compact]
    raise ValueError(f"{label}: '{raw}' is not a valid category. Choose a number from the list or type one of the shown values.")

def input_number(label, default=None, minimum=None, maximum=None, integer=False):
    while True:
        default_text = f' [default: {default}]' if default is not None and pd.notna(default) else ''
        print('\n' + '=' * 72)
        print(f'LABEL : {label}')
        print('TYPE  : INTEGER' if integer else 'TYPE  : NUMBER')
        print(f'ANSWER TO TYPE{default_text}')
        raw = input('> ').strip()
        if raw == '' and default is not None and pd.notna(default):
            value = float(default)
        else:
            try:
                value = float(raw)
            except Exception:
                print('Please type a number, for example 2 or 10.5.')
                continue
        if minimum is not None and value < minimum:
            print(f'Value must be >= {minimum}.')
            continue
        if maximum is not None and value > maximum:
            print(f'Value must be <= {maximum}.')
            continue
        if integer:
            if not float(value).is_integer():
                print('Please enter a whole number.')
                continue
            return int(value)
        return value

def input_category(label, allowed, default=None):
    allowed = list(allowed)
    while True:
        print('\n' + '=' * 72)
        print(f'LABEL : {label}')
        print('TYPE  : STRING / CATEGORY (case-insensitive)')
        print('OPTIONS — enter the NUMBER or type the VALUE:')
        for i, value in enumerate(allowed, 1):
            marker = '  <-- DEFAULT' if default is not None and str(value) == str(default) else ''
            print(f'  {i:3d}. {value}{marker}')
        if default is not None and pd.notna(default):
            print(f'ANSWER TO TYPE [press Enter for default: {default}]')
        else:
            print('ANSWER TO TYPE [number or category value]')
        raw = input('> ').strip()
        if raw == '' and default is not None and pd.notna(default):
            return canonical_category(label, default, allowed)
        if raw.isdigit() and 1 <= int(raw) <= len(allowed):
            return allowed[int(raw) - 1]
        try:
            return canonical_category(label, raw, allowed)
        except ValueError as e:
            print(e)

def latest_default(label, latest_profile):
    if label in latest_profile.index and pd.notna(latest_profile[label]):
        return latest_profile[label]
    return None

def allowed_values(label):
    if label in df.columns:
        return sorted(df[label].dropna().astype(str).drop_duplicates().tolist(), key=lambda x: x.lower())
    return []


In [22]:
# 6. SELECT CITY — HOUSE REMOVED COMPLETELY
cities = sorted(df[CITY_COL].dropna().astype(str).drop_duplicates().tolist(), key=lambda x: x.lower())
USER_CITY = input_category(CITY_COL, cities, default=cities[0] if cities else None)

city_history = df[
    df[CITY_COL].astype(str).str.casefold() == str(USER_CITY).casefold()
].sort_values(DATE_COL).copy()
if city_history.empty:
    raise ValueError('No historical records found for the selected City.')

# Use the latest city-level profile; no House selection or identifier is used.
latest_profile = city_history.iloc[-1]

print('\n' + '=' * 72)
print('SELECTED LOCATION')
print('City:', USER_CITY)
print('House: REMOVED')
print('Latest historical date:', latest_profile[DATE_COL])
print('=' * 72)



LABEL : city
TYPE  : STRING / CATEGORY (case-insensitive)
OPTIONS — enter the NUMBER or type the VALUE:
    1. Islamabad  <-- DEFAULT
    2. Karachi
    3. Lahore
    4. Multan
    5. Peshawar
    6. Skardu
ANSWER TO TYPE [press Enter for default: Islamabad]

SELECTED LOCATION
City: Islamabad
House: REMOVED
Latest historical date: 2024-10-31 00:00:00


In [23]:
# 7. GET FORECAST DATE AND HORIZON
default_start = df[DATE_COL].max() + pd.Timedelta(days=1)

while True:
    print('\n' + '=' * 72)
    print('LABEL : Forecast start date')
    print('TYPE  : DATE (YYYY-MM-DD)')
    print(f'ANSWER TO TYPE [default: {default_start.date()}]')
    raw = input('> ').strip()
    try:
        START_DATE = pd.Timestamp(raw) if raw else default_start
        if START_DATE <= df[DATE_COL].max():
            raise ValueError('Start date must be after the latest historical date.')
        break
    except Exception as e:
        print('Invalid date:', e)

print('\n' + '=' * 72)
print('LABEL : Forecast horizon')
print('TYPE  : INTEGER')
print('OPTIONS: 1 = one day | 7 = seven days | 30 = thirty days')
print('ANSWER TO TYPE [default: 7]')
while True:
    raw = input('> ').strip() or '7'
    try:
        HORIZON_DAYS = int(raw)
        if HORIZON_DAYS in (1, 7, 30):
            break
        print('Please enter 1, 7, or 30.')
    except ValueError:
        print('Please enter 1, 7, or 30.')



LABEL : Forecast start date
TYPE  : DATE (YYYY-MM-DD)
ANSWER TO TYPE [default: 2024-11-28]



LABEL : Forecast horizon
TYPE  : INTEGER
OPTIONS: 1 = one day | 7 = seven days | 30 = thirty days
ANSWER TO TYPE [default: 7]


In [24]:
# 8. USER INPUT — FEWER, NECESSARY FEATURES ONLY
# The user enters only the selected practical features.
# All other trained model features remain automatic.

if "schema_df" not in globals():
    raise RuntimeError("Run Cell 3 (BUILD AND SAVE THE COMPLETE INPUT SCHEMA) first.")

if "label" not in schema_df.columns:
    raise RuntimeError("schema_df does not contain 'label'. Re-run Cell 3.")

USER_INPUTS = {CITY_COL: USER_CITY}
input_rows = []

# Calendar features are NEVER user-entered.
calendar_names = {
    "year", "month", "day", "day_of_week",
    "week_of_year", "day_of_year", "is_weekend", "season"
}

def get_meta(label):
    match = schema_df[schema_df["label"].astype(str) == str(label)]
    if match.empty:
        raise KeyError(f"Feature '{label}' is not present in schema_df. Re-run Cell 3.")
    return match.iloc[0]

for label in USER_FEATURES:
    meta = get_meta(label)
    model_type = str(meta["model_type"])
    default = latest_default(label, latest_profile)

    if model_type == "categorical":
        allowed = allowed_values(label)
        if allowed:
            value = input_category(label, allowed, default=default)
        else:
            print("\n" + "=" * 72)
            print(f"LABEL : {label}")
            print("TYPE  : STRING / CATEGORY")
            print("ANSWER TO TYPE: enter a text value")
            raw = input("> ").strip()
            value = raw if raw else default
    else:
        lname = norm_name(label)
        integer = (
            label in appliance_columns
            or any(k in lname for k in [
                "numberof", "noof", "people", "residents",
                "floors", "rooms", "washrooms", "stores",
                "children", "adults", "seniors"
            ])
        )
        value = input_number(
            label,
            default=default,
            minimum=0 if integer else None,
            integer=integer
        )

    USER_INPUTS[label] = value
    input_rows.append({
        "label": label,
        "model_type": model_type,
        "pandas_dtype": meta["pandas_dtype"],
        "python_input_type": meta["python_input_type"],
        "value_type": meta["value_type"],
        "user_value": value,
        "source": "user input" if str(value) != str(default) else "default / latest profile",
    })

user_input_df = pd.DataFrame(input_rows)
USER_INPUT_FILE = PROCESSED_DIR / "powerplus_user_input_values.csv"
user_input_df.to_csv(USER_INPUT_FILE, index=False)

print("\n" + "=" * 72)
print("USER INPUT SUMMARY")
print("=" * 72)
print(f"User-entered feature fields: {len(user_input_df)}")
print(f"Automatic model features   : {len(feature_columns) - len(USER_FEATURES)}")
print(f"Saved answers              : {USER_INPUT_FILE}")
display(user_input_df)



LABEL : No. of people (Temp+Perm)
TYPE  : INTEGER
ANSWER TO TYPE [default: 2.0]

LABEL : Covered Area
TYPE  : NUMBER
ANSWER TO TYPE [default: 5.0]

LABEL : No. of rooms
TYPE  : INTEGER
ANSWER TO TYPE [default: 4.0]

LABEL : house_age
TYPE  : NUMBER
ANSWER TO TYPE [default: 3.0]

LABEL : Ceiling Type
TYPE  : STRING / CATEGORY (case-insensitive)
OPTIONS — enter the NUMBER or type the VALUE:
    1. Cemented  <-- DEFAULT
    2. Concrete
    3. Fal Ceiling
    4. Fall Ceiling
    5. Fall Celling
    6. Painted
    7. Wooden
ANSWER TO TYPE [press Enter for default: Cemented]

LABEL : Roof Type
TYPE  : STRING / CATEGORY (case-insensitive)
OPTIONS — enter the NUMBER or type the VALUE:
    1. Cemented  <-- DEFAULT
    2. Cemeted
    3. Concrete + reflected paint in july
    4. Mud Plaster
    5. Solar Covered
    6. solar Panel
ANSWER TO TYPE [press Enter for default: Cemented]
Roof Type: 'Wooden' is not a valid category. Choose a number from the list or type one of the shown values.

LABEL : 

,label,model_type,pandas_dtype,python_input_type,value_type,user_value,source
0,No. of people (Temp+Perm),numeric,float64,float,number,7,user input
1,Covered Area,numeric,float64,float,number,1500.0,user input
2,No. of rooms,numeric,float64,float,number,7,user input
3,house_age,numeric,float64,float,number,10.0,user input
4,Ceiling Type,categorical,object,str,string/category,Cemented,default / latest profile
5,Roof Type,categorical,object,str,string/category,Cemeted,user input
6,Air Conditioners,numeric,float64,float,number,2,user input
7,Air Coolers,numeric,int64,float,number,2,user input
8,Refrigerators,numeric,float64,float,number,2,user input
9,Washing Machines,numeric,int64,float,number,2,user input


In [25]:
# 9. HISTORICAL DEMAND + DAILY WEATHER HELPERS
LAG_MAP = {lag: next((c for c in [f'lag_{lag}_day_kwh', f'Lag_{lag}', f'lag_{lag}'] if c in feature_columns), None) for lag in [1,2,3,7,14,30]}
ROLLING_MAP = {w: next((c for c in [f'rolling_{w}_day_avg_kwh', f'Rolling_{w}', f'rolling_{w}'] if c in feature_columns), None) for w in [3,7,14,30]}

# Use the weather columns actually present in the trained model.
WEATHER_COLUMNS = [c for c in feature_columns if c in set(weather_value_cols)]

def season_from_month(month):
    return 'Winter' if month in [12,1,2] else 'Spring' if month in [3,4,5] else 'Summer' if month in [6,7,8] else 'Autumn'

def future_weather(city, dates):
    """Use exact weather when available; otherwise use city/month historical means."""
    hist = df.copy()
    hist['_month'] = hist[DATE_COL].dt.month
    city_hist = hist[hist[CITY_COL].astype(str).str.strip().str.casefold() == str(city).strip().casefold()]
    rows = []
    for d in dates:
        exact = city_hist[city_hist[DATE_COL].dt.normalize() == pd.Timestamp(d).normalize()]
        month_hist = city_hist[city_hist['_month'] == d.month]
        source = exact if not exact.empty else month_hist
        row = {DATE_COL: pd.Timestamp(d), CITY_COL: city}
        for c in WEATHER_COLUMNS:
            vals = pd.to_numeric(source[c], errors='coerce').dropna()
            fallback = pd.to_numeric(city_hist[c], errors='coerce').median() if c in city_hist else np.nan
            row[c] = float(vals.mean()) if len(vals) else fallback
        rows.append(row)
    return pd.DataFrame(rows)

print('Lag features:', LAG_MAP)
print('Rolling features:', ROLLING_MAP)
print('Weather features used by model:', WEATHER_COLUMNS)


Lag features: {1: 'lag_1_day_kwh', 2: 'lag_2_day_kwh', 3: 'lag_3_day_kwh', 7: 'lag_7_day_kwh', 14: 'lag_14_day_kwh', 30: 'lag_30_day_kwh'}
Rolling features: {3: 'rolling_3_day_avg_kwh', 7: 'rolling_7_day_avg_kwh', 14: 'rolling_14_day_avg_kwh', 30: 'rolling_30_day_avg_kwh'}
Weather features used by model: ['temperature_mean_weather', 'humidity_mean_weather', 'dew_mean_weather', 'wind_speed_mean_weather', 'wind_direction_mean_weather', 'pressure_mean_weather', 'solar_radiation_mean_weather', 'temperature_max_weather', 'humidity_max_weather', 'wind_speed_max_weather', 'solar_radiation_max_weather', 'uv_index_max_weather', 'precipitation_sum_weather', 'solar_energy_sum_weather']


In [26]:
# 10. BUILD FUTURE MODEL ROW
def build_future_row(future_date, demand_history, weather_row=None):
    row = latest_profile.to_dict()
    row[CITY_COL] = USER_CITY

    # Apply user inputs first.
    for c, v in USER_INPUTS.items():
        if c in feature_columns and v is not None:
            row[c] = v

    # Calendar features are refreshed for each forecast day.
    calendar = {
        'year': future_date.year,
        'month': future_date.month,
        'day': future_date.day,
        'day_of_week': future_date.dayofweek,
        'week_of_year': int(future_date.isocalendar().week),
        'day_of_year': future_date.dayofyear,
        'is_weekend': int(future_date.dayofweek >= 5),
        'season': season_from_month(future_date.month),
    }
    for c, v in calendar.items():
        if c in feature_columns:
            row[c] = v

    # Keep total appliance count consistent when the feature exists.
    if 'total_appliance_count' in feature_columns and ALL_APPLIANCE_COLUMNS:
        total = 0.0
        for c in ALL_APPLIANCE_COLUMNS:
            try:
                total += float(row.get(c, 0)) if pd.notna(row.get(c, 0)) else 0.0
            except Exception:
                pass
        row['total_appliance_count'] = total

    # Refresh house age if build year is available.
    build_year_candidates = [c for c in feature_columns if norm_name(c) in {'buildyearofhouse','buildyear'}]
    if 'house_age' in feature_columns and build_year_candidates:
        by = row.get(build_year_candidates[0])
        if pd.notna(by):
            row['house_age'] = max(0, future_date.year - float(by))

    if weather_row is not None:
        for c in WEATHER_COLUMNS:
            if c in feature_columns and c in weather_row.index and pd.notna(weather_row[c]):
                row[c] = weather_row[c]

    # Recursive lag values.
    for lag, col in LAG_MAP.items():
        if col is None:
            continue
        target_date = future_date - pd.Timedelta(days=lag)
        m = demand_history[demand_history[DATE_COL] == target_date]
        if not m.empty:
            row[col] = float(m[TARGET].iloc[-1])
        elif future_date == START_DATE and col in USER_INPUTS and USER_INPUTS[col] is not None:
            row[col] = float(USER_INPUTS[col])
        else:
            row[col] = float(demand_history[TARGET].iloc[-1])

    past = demand_history[demand_history[DATE_COL] < future_date].sort_values(DATE_COL)[TARGET]
    for window, col in ROLLING_MAP.items():
        if col is None:
            continue
        if len(past) >= window:
            row[col] = float(past.tail(window).mean())
        elif future_date == START_DATE and col in USER_INPUTS and USER_INPUTS[col] is not None:
            row[col] = float(USER_INPUTS[col])
        elif len(past):
            row[col] = float(past.mean())

    # Enforce expected model types.
    for c in numeric_features:
        if c in row:
            row[c] = pd.to_numeric(pd.Series([row[c]]), errors='coerce').iloc[0]
    for c in categorical_features:
        if c in row and pd.notna(row[c]):
            row[c] = str(row[c])

    return pd.DataFrame([row]).reindex(columns=feature_columns)


In [27]:
# 11. RUN XGBOOST PREDICTION
def predict_for_user(horizon_days):
    dates = pd.date_range(START_DATE, periods=horizon_days, freq='D')
    demand = city_history.groupby(DATE_COL, as_index=False)[TARGET].mean().dropna()
    weather_proxy = future_weather(USER_CITY, dates)
    results = []

    for d in dates:
        wr = None
        if not weather_proxy.empty:
            m = weather_proxy[weather_proxy[DATE_COL] == d]
            if not m.empty:
                wr = m.iloc[0]
        X_future = build_future_row(d, demand, weather_row=wr)
        pred = max(0.0, float(pipeline.predict(X_future)[0]))
        results.append({
            'date': d,
            'city': USER_CITY,
            'predicted_electricity_kwh': pred,
        })
        demand = pd.concat([demand, pd.DataFrame([{DATE_COL: d, TARGET: pred}])], ignore_index=True)
    return pd.DataFrame(results)

forecast_1 = predict_for_user(1)
forecast_7 = predict_for_user(7)
forecast_30 = predict_for_user(30)

print('\n' + '=' * 72)
print('XGBOOST USER INPUT PREDICTION OUTPUT')
print('=' * 72)
print('City                 :', USER_CITY)
print('Forecast start       :', START_DATE.date())
print('1-day predicted kWh  :', round(forecast_1['predicted_electricity_kwh'].sum(), 4))
print('7-day predicted kWh  :', round(forecast_7['predicted_electricity_kwh'].sum(), 4))
print('30-day predicted kWh :', round(forecast_30['predicted_electricity_kwh'].sum(), 4))

print('\n1-DAY OUTPUT')
display(forecast_1)
print('\n7-DAY OUTPUT')
display(forecast_7)
print('\n30-DAY OUTPUT')
display(forecast_30)



XGBOOST USER INPUT PREDICTION OUTPUT
City                 : Islamabad
Forecast start       : 2026-09-09
1-day predicted kWh  : 0.0243
7-day predicted kWh  : 0.17
30-day predicted kWh : 0.7285

1-DAY OUTPUT


,date,city,predicted_electricity_kwh
0,2026-09-09,Islamabad,0.024284



7-DAY OUTPUT


,date,city,predicted_electricity_kwh
0,2026-09-09,Islamabad,0.024284
1,2026-09-10,Islamabad,0.024284
2,2026-09-11,Islamabad,0.024284
3,2026-09-12,Islamabad,0.024284
4,2026-09-13,Islamabad,0.024284
5,2026-09-14,Islamabad,0.024284
6,2026-09-15,Islamabad,0.024284



30-DAY OUTPUT


,date,city,predicted_electricity_kwh
0,2026-09-09,Islamabad,0.024284
1,2026-09-10,Islamabad,0.024284
2,2026-09-11,Islamabad,0.024284
3,2026-09-12,Islamabad,0.024284
4,2026-09-13,Islamabad,0.024284
5,2026-09-14,Islamabad,0.024284
6,2026-09-15,Islamabad,0.024284
7,2026-09-16,Islamabad,0.024284
8,2026-09-17,Islamabad,0.024284
9,2026-09-18,Islamabad,0.024284


In [28]:
# 12. APPLIANCE-LEVEL WHAT-IF ANALYSIS
# This estimates the change in TOTAL household predicted kWh when one appliance count increases by 1.
# It is model sensitivity, not measured appliance-specific consumption.
BASELINE = float(forecast_1['predicted_electricity_kwh'].iloc[0])
sensitivity_rows = []

for appliance in appliance_columns:
    base = USER_INPUTS.get(appliance, latest_default(appliance, latest_profile))
    try:
        base = 0.0 if pd.isna(base) else float(base)
    except Exception:
        continue

    USER_INPUTS[appliance] = base + 1
    changed_pred = float(predict_for_user(1)['predicted_electricity_kwh'].iloc[0])
    USER_INPUTS[appliance] = base

    sensitivity_rows.append({
        'appliance': appliance,
        'base_count': base,
        'count_plus_1': base + 1,
        'baseline_total_kwh': BASELINE,
        'plus_1_total_kwh': changed_pred,
        'model_delta_kwh': changed_pred - BASELINE,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
if not sensitivity_df.empty:
    sensitivity_df = sensitivity_df.sort_values('model_delta_kwh', ascending=False)
display(sensitivity_df)


,appliance,base_count,count_plus_1,baseline_total_kwh,plus_1_total_kwh,model_delta_kwh
0,Air Conditioners,2.0,3.0,0.024284,0.024284,0.0
1,Air Coolers,2.0,3.0,0.024284,0.024284,0.0
2,Refrigerators,2.0,3.0,0.024284,0.024284,0.0
3,Washing Machines,2.0,3.0,0.024284,0.024284,0.0
4,Celling Fans,2.0,3.0,0.024284,0.024284,0.0
5,Water Pumps,2.0,3.0,0.024284,0.024284,0.0
6,Electric heaters,2.0,3.0,0.024284,0.024284,0.0
7,Electric Cooker,2.0,3.0,0.024284,0.024284,0.0
8,Geysers,2.0,3.0,0.024284,0.024284,0.0
9,LED Bulbs,2.0,3.0,0.024284,0.024284,0.0


In [29]:
# 13. SAVE ALL OUTPUTS FOR POWER BI
OUT_1 = PROCESSED_DIR / 'powerplus_user_input_forecast_1_day.csv'
OUT_7 = PROCESSED_DIR / 'powerplus_user_input_forecast_7_day.csv'
OUT_30 = PROCESSED_DIR / 'powerplus_user_input_forecast_30_day.csv'
OUT_SENS = PROCESSED_DIR / 'powerplus_appliance_sensitivity.csv'

forecast_1.to_csv(OUT_1, index=False)
forecast_7.to_csv(OUT_7, index=False)
forecast_30.to_csv(OUT_30, index=False)
sensitivity_df.to_csv(OUT_SENS, index=False)

print('Saved prediction outputs:')
for p in [SCHEMA_FILE, USER_INPUT_FILE, OUT_1, OUT_7, OUT_30, OUT_SENS]:
    print(' -', p)


Saved prediction outputs:
 - d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_user_input_schema.csv
 - d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_user_input_values.csv
 - d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_user_input_forecast_1_day.csv
 - d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_user_input_forecast_7_day.csv
 - d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_user_input_forecast_30_day.csv
 - d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_appliance_sensitivity.csv


In [30]:
# 14. FINAL USER INPUT PREDICTION OUTPUT

print("\n" + "=" * 80)
print("POWERPLUS — XGBOOST USER INPUT PREDICTION OUTPUT (HOUSE REMOVED FROM TRAINING AND USER PREDICTION)")
print("=" * 80)
print(f"City              : {USER_CITY}")
print(f"Forecast start    : {START_DATE.date()}")
print(f"Horizon requested : {HORIZON_DAYS} day(s)")
print("-" * 80)

print(f"1-Day predicted household demand  : {forecast_1['predicted_electricity_kwh'].sum():.4f} kWh")
print(f"7-Day predicted household demand  : {forecast_7['predicted_electricity_kwh'].sum():.4f} kWh")
print(f"30-Day predicted household demand : {forecast_30['predicted_electricity_kwh'].sum():.4f} kWh")

print("\nSELECTED USER INPUTS")
display(
    user_input_df[
        ["label", "model_type", "pandas_dtype",
         "python_input_type", "user_value"]
    ]
)

print("\nPREDICTION OUTPUT")
if HORIZON_DAYS == 1:
    display(forecast_1)
elif HORIZON_DAYS == 7:
    display(forecast_7)
else:
    display(forecast_30)



POWERPLUS — XGBOOST USER INPUT PREDICTION OUTPUT (HOUSE REMOVED FROM TRAINING AND USER PREDICTION)
City              : Islamabad
Forecast start    : 2026-09-09
Horizon requested : 30 day(s)
--------------------------------------------------------------------------------
1-Day predicted household demand  : 0.0243 kWh
7-Day predicted household demand  : 0.1700 kWh
30-Day predicted household demand : 0.7285 kWh

SELECTED USER INPUTS


,label,model_type,pandas_dtype,python_input_type,user_value
0,No. of people (Temp+Perm),numeric,float64,float,7
1,Covered Area,numeric,float64,float,1500.0
2,No. of rooms,numeric,float64,float,7
3,house_age,numeric,float64,float,10.0
4,Ceiling Type,categorical,object,str,Cemented
5,Roof Type,categorical,object,str,Cemeted
6,Air Conditioners,numeric,float64,float,2
7,Air Coolers,numeric,int64,float,2
8,Refrigerators,numeric,float64,float,2
9,Washing Machines,numeric,int64,float,2



PREDICTION OUTPUT


,date,city,predicted_electricity_kwh
0,2026-09-09,Islamabad,0.024284
1,2026-09-10,Islamabad,0.024284
2,2026-09-11,Islamabad,0.024284
3,2026-09-12,Islamabad,0.024284
4,2026-09-13,Islamabad,0.024284
5,2026-09-14,Islamabad,0.024284
6,2026-09-15,Islamabad,0.024284
7,2026-09-16,Islamabad,0.024284
8,2026-09-17,Islamabad,0.024284
9,2026-09-18,Islamabad,0.024284


## How to use

1. Run the cells from top to bottom.
2. Select **City** using the numbered options.
3. Enter the **Forecast start date**.
4. Enter only the selected **home/household** and **key equipment** fields.
5. For categorical fields, enter the option number or type the category value; letter case does not matter.
6. The notebook automatically supplies the remaining trained XGBoost features from the selected city profile/history and derives calendar/lag/rolling features.
7. Run the prediction cells to generate 1-day, 7-day, and 30-day forecasts.
8. Run Cell 13 to save forecast CSVs and Cell 16 to create **`powerplus_prediction_summary.csv`** for Power BI.

### Power BI CSV files
- `powerplus_user_input_forecast_7_day.csv` — daily 7-day predictions.
- `powerplus_user_input_forecast_30_day.csv` — daily 30-day predictions.
- `powerplus_prediction_summary.csv` — KPI values for cards.
- `powerplus_user_input_values.csv` — the user's reduced input values.
- `powerplus_appliance_sensitivity.csv` — what-if sensitivity of total household demand to appliance counts.
- `powerplus_user_input_forecast_1_day.csv` — first-day prediction.


In [31]:
# 16. CREATE POWER BI PREDICTION SUMMARY CSV
# This cell uses the forecasts generated by THIS notebook.
# It creates one compact row for Power BI KPI cards.

from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path(PROCESSED_DIR)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if 'forecast_7' not in globals() or 'forecast_30' not in globals():
    raise RuntimeError('Run the forecast/prediction cells before creating the Power BI summary.')


def prediction_series(forecast_df):
    for col in [
        'predicted_electricity_kwh', 'predicted_kwh',
        'Predicted_KWh', 'prediction', 'Prediction',
        'electricity_kwh'
    ]:
        if col in forecast_df.columns:
            return pd.to_numeric(forecast_df[col], errors='coerce').dropna()
    raise ValueError(f'Prediction column not found. Columns: {forecast_df.columns.tolist()}')


values_7 = prediction_series(forecast_7)
values_30 = prediction_series(forecast_30)

if len(values_7) == 0 or len(values_30) == 0:
    raise ValueError('Forecast output contains no numeric predictions.')

prediction_id = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

prediction_summary = pd.DataFrame([{
    'Prediction_ID': prediction_id,
    'City': USER_CITY,
    'Forecast_Start': pd.to_datetime(START_DATE),
    'Forecast_7_Day_Total': values_7.sum(),
    'Forecast_30_Day_Total': values_30.sum(),
    'Average_Daily_KWh': values_30.mean(),
    'Peak_Daily_KWh': values_30.max(),
    'Minimum_Daily_KWh': values_30.min(),
    'User_Feature_Count': len(USER_FEATURES),
    'Model_Feature_Count': len(feature_columns),
}])

SUMMARY_FILE = PROCESSED_DIR / 'powerplus_prediction_summary.csv'
prediction_summary.to_csv(SUMMARY_FILE, index=False)

print('=' * 72)
print('POWER BI PREDICTION SUMMARY CREATED')
print('=' * 72)
display(prediction_summary)
print(f'Saved to: {SUMMARY_FILE}')


POWER BI PREDICTION SUMMARY CREATED


,Prediction_ID,City,Forecast_Start,Forecast_7_Day_Total,Forecast_30_Day_Total,Average_Daily_KWh,Peak_Daily_KWh,Minimum_Daily_KWh,User_Feature_Count,Model_Feature_Count
0,20260908_173100,Islamabad,2026-09-09,0.169988,0.72852,0.024284,0.024284,0.024284,16,95


Saved to: d:\Project-Electricity-Demand-Forecasting\processed_data\powerplus_prediction_summary.csv
